# 03 Behavioral Monitoring and BPoW
Shows behavioral vector extraction, supervised and unsupervised risk scoring, proof-of-work solving, and anomaly reporting using the shared package modules.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import hashlib
from src.behavioral.anomaly import UserBehavioralProfile
from src.behavioral.extractor import BehavioralSession
from src.behavioral.models import get_default_model_suite
from src.behavioral.pow import assess_pow_risk, compute_difficulty, solve_pow, verify_pow

session = BehavioralSession('user-a')
session.record_chunk(b'A' * 4096, timestamp_ms=0.0)
session.record_chunk(b'B' * 4096, timestamp_ms=120.0)
vector = session.extract_vector()
ku_public = hashlib.sha256(b'ku-public').digest()
nonce, proof_hash = solve_pow(vector, session.session_id, 1, ku_public)
profile = UserBehavioralProfile('user-a')
for tau in [90.0, 100.0, 110.0]:
    profile.update({
        'tau_avg': tau,
        'tau_std': 4.0,
        'tau_min': tau - 4.0,
        'tau_max': tau + 4.0,
        'interarrival_cv': 0.04,
        'tau_seq_hash': hashlib.sha256(f'tau:{tau}'.encode('utf-8')).hexdigest(),
        'entropy_mean': 7.0,
        'entropy_std': 0.2,
        'entropy_min': 6.8,
        'entropy_max': 7.2,
        'entropy_dist_hash': hashlib.sha256(b'entropy').hexdigest(),
        'chunk_order_hash': hashlib.sha256(b'order').hexdigest(),
        'n_chunks': 2,
    })
model_suite = get_default_model_suite()
risk_report = assess_pow_risk(vector)
{
    'model_backend': model_suite.backend,
    'difficulty': risk_report['effective_difficulty'],
    'pow_valid': verify_pow(vector, session.session_id, 1, ku_public, nonce, compute_difficulty(vector)),
    'proof_hash': proof_hash[:16],
    'anomaly_flags': profile.anomaly_report(vector)['flags'],
    'supervised_prediction': risk_report['supervised_prediction'],
    'supervised_human_probability': round(risk_report['supervised_human_probability'], 3),
    'unsupervised_prediction': risk_report['unsupervised_prediction'],
    'unsupervised_score': round(risk_report['unsupervised_score'], 3),
}


{'model_backend': 'sklearn',
 'difficulty': 16,
 'pow_valid': True,
 'proof_hash': '0000d181c2100def',
 'anomaly_flags': [],
 'supervised_prediction': 'human',
 'supervised_human_probability': 0.883,
 'unsupervised_prediction': 'outlier',
 'unsupervised_score': -0.089}